In [1]:
import pandas as pd
import yfinance as yf

Some tickers are not recognized by yfinance (usually non-US stocks) and need to be modified to add the country suffix at the end of the ticker. In order to optimize the tests, we use patterns to try to deduce ASAP the quotation country.

In [2]:
# 1. Read CSV while forcing the ticker column to stay as text (keeps leading zeros)
df = pd.read_csv("../../data/processed/stocks/stocks_cleaned.csv", dtype={"ticker": str})
tickers = df["ticker"].dropna().tolist()

In [3]:
def find_valid_ticker(raw_ticker):
    
    # 2. Find the tickers working with yfinance library.
    ticker = str(raw_ticker).strip().upper()
    candidates = []

    # Check numeric-only tickers based on digit length
    if ticker.isdigit():
        digit_count = len(ticker)

        if digit_count == 4:
            # 4-digit tickers: Try 4-digit markets first (Japan, Taiwan, Hong Kong)
            ticker_4 = ticker.zfill(4)
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan TWSE
            candidates.append(ticker_4 + ".TWO") # Taiwan TPEx
            candidates.append(ticker_4 + ".HK")  # Hong Kong

            # Fallback: Zero-pad to 6 digits for Korea / China
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_6 + ".KS")  # Korea KOSPI
            candidates.append(ticker_6 + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker_6 + ".SS")  # Shanghai
            candidates.append(ticker_6 + ".SZ")  # Shenzhen

        elif digit_count == 6:
            # 6-digit tickers: Try 6-digit markets FIRST (Korea, China)
            candidates.append(ticker + ".KS")  # Korea KOSPI
            candidates.append(ticker + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker + ".SS")  # Shanghai
            candidates.append(ticker + ".SZ")  # Shenzhen

            # Fallback: Trim to 4 digits for Japan / Taiwan
            ticker_4 = ticker[-4:]
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan
            candidates.append(ticker_4 + ".HK")  # Hong Kong

        else:
            # 5-digit or other lengths (e.g. Hong Kong 1810 / 9880)
            ticker_4 = ticker.zfill(4)
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_4 + ".HK")
            candidates.append(ticker_6 + ".KS")
            candidates.append(ticker_6 + ".SS")
            candidates.append(ticker_6 + ".SZ")

    else:
        # Non-numeric tickers (US & popular European tickers)
        candidates.append(ticker)             # US
        candidates.append(ticker + ".DE")     # Germany
        candidates.append(ticker + ".F")      # Frankfurt
        candidates.append(ticker + ".L")      # London
        candidates.append(ticker + ".PA")     # Paris
        candidates.append(ticker + ".SW")     # Switzerland

    # Remove duplicates while keeping exact order of priority
    candidates = list(dict.fromkeys(candidates))

    # Test candidate tickers to see if a yfinance command works (valid ticker):
    for candidate in candidates:
        try:
            hist = yf.Ticker(candidate).history(period="1d")
            if not hist.empty:
                print(f"SUCCESS: {raw_ticker} -> {candidate}")
                return candidate
        except Exception:
            pass

    print(f"NOT FOUND: {raw_ticker}")


In [4]:
# 2. run over all tickers and collect results
results = {}
not_found_tickers = []

for t in tickers:
    matched = find_valid_ticker(t)
    results[t] = matched
    
    # Collect missing tickers into a list
    if matched is None:
        not_found_tickers.append(t)

print("Finished processing all tickers.")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 000333.KS"}}}
$000333.KS: No data found, symbol may be delisted
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 000333.KQ"}}}
$000333.KQ: No data found, symbol may be delisted
$000333.SS: No data found, symbol may be delisted


SUCCESS: 000333 -> 000333.SZ


$002050.KS: No data found, symbol may be delisted
$002050.KQ: No data found, symbol may be delisted
$002050.SS: No data found, symbol may be delisted


SUCCESS: 002050 -> 002050.SZ


$002230.KS: No data found, symbol may be delisted


...

SUCCESS: RRX -> RRX


$SHA.DE: No data found, symbol may be delisted
$SHA.F: No data found, symbol may be delisted
$SHA.L: No data found, symbol may be delisted
$SHA.PA: No data found, symbol may be delisted
$SHA.SW: No data found, symbol may be delisted


NOT FOUND: SHA


$SIE: No data found, symbol may be delisted


SUCCESS: SIE -> SIE.DE
SUCCESS: SONY -> SONY
SUCCESS: SQM -> SQM
SUCCESS: STM -> STM
SUCCESS: TM -> TM
SUCCESS: TMV -> TMV
SUCCESS: TRMB -> TRMB
SUCCESS: TSLA -> TSLA
SUCCESS: TXN -> TXN
SUCCESS: U -> U
SUCCESS: USAR -> USAR
SUCCESS: UUUU -> UUUU
SUCCESS: VPG -> VPG
SUCCESS: WMT -> WMT
SUCCESS: WOLF -> WOLF
SUCCESS: XPEV -> XPEV
Finished processing all tickers.


In [5]:
# 3. Print the missing tickers summary at the end
print(f"SUMMARY: {len(not_found_tickers)} ticker(s) could not be resolved:")
if not_found_tickers:
    print(not_found_tickers)
else:
    print("All tickers were successfully matched!")

SUMMARY: 2 ticker(s) could not be resolved:
['MOGA', 'SHA']


In [6]:
# 4. Update ticker column with resolved tickers
df["ticker"] = df["ticker"].replace(results)

# Remove rows where no valid ticker was found
df = df.dropna(subset=["ticker"])
df

,ticker,layer
0,000333.SZ,Layer 11
1,002050.SZ,Layer 9
2,002230.KQ,Layer 3
3,002460.KS,Layer 1
4,002594.SZ,Layer 6
...,...,...
97,UUUU,Layer 1
98,VPG,Layer 7
99,WMT,Layer 12
100,WOLF,Layer 6


# Export

In [7]:
# Export
df.to_csv("../../data/processed/stocks/stocks_transformed_verified.csv", index=False)